# gramlab — stage 2: finish the 25M writer (fp16), BM25 retrieval + retrieval-conditioned Q&A, retrieval scoreboard, teacher targets for the meaning map, per-POS chart

**Save Version → Save & Run All**, GPU **T4 x2**, Internet **on**. Attach the overnight notebook's **Version 1** output as an input
(*Add Input → Your Work → Notebooks*). Only what is still needed is copied from it: tokenizer + shards, the `small_wiki`
checkpoint (with optimizer, to resume), the two tiny checkpoints (weights only, for the per-POS chart).

Cost: full mode **≈5–5.5 GPU-hours** of the weekly ~30 (ceiling `TRAIN_HOURS` + 1.5 h); `QUICK = True` **≈0.75 h** — run it once first,
the torch paths have never executed. Output ≈ 3 GB and it is the new base: raw Wikipedia text is kept so later runs never re-download,
`work/` holds shards, checkpoints, the BM25 index and the teacher embeddings. Dropped from the pipeline: grammar-head retraining and
BLiMP (parked after the null result), the fresh-start fallback. The chat cell (§9) is back, self-contained, so the current model can be tried by hand. See RESULTS.md.

## Parameters

In [ ]:
QUICK = False        # True = ~45-min smoke test of every stage (300 steps, 5k Q&A dialogues, 400 scoreboard queries)
TRAIN_HOURS = 4.0    # wall-clock budget for the continued pretraining; the cosine schedule is fitted into it after 200 measured steps
MORE_STEPS = 24000   # fp16 measured at 0.52 s/step on the T4 (quick run): ~24000 steps fit in 4 h = ~790M tokens, ~3.6 more epochs of the 217M mix (~4.2 total)
SWEEP = True          # sample-efficiency sweep: baseline vs grammar heads at 250k / 1M / 4M tokens x 3 seeds (~45 min). The test run 1 skipped.
# Q&A-only rerun (fix the fine-tune, keep the trained writer): MORE_STEPS = 0, SWEEP = False, attach the latest stage-2 output; ~40 min.


## 0. Code, corpora, and the pieces of Version 1 that are still needed

In [ ]:
import os, subprocess, sys, time, glob, shutil, json
T0 = time.time()
def sh(cmd, must=True):
    print('\n$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if must and r.returncode != 0:
        raise SystemExit(f'command failed ({r.returncode}): {cmd}')
    return r.returncode

os.chdir('/kaggle/working')
if not os.path.exists('gramlab'):
    sh('git clone --depth 1 https://github.com/furevych/gramlab.git')
else:
    sh('git -C gramlab pull --ff-only', must=False)
os.chdir('/kaggle/working/gramlab')
os.makedirs('external', exist_ok=True); os.makedirs('results', exist_ok=True)
for url, d in [('https://github.com/gucorpling/amalgum', 'external/amalgum'),
               ('https://github.com/UniversalDependencies/UD_English-GUM', 'external/UD_English-GUM'),
               ('https://github.com/alexwarstadt/blimp', 'external/blimp')]:
    if not os.path.exists(d):
        sh(f'GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 {url} {d}')

prev = sorted(glob.glob('/kaggle/input/**/work/runs/small_wiki/ckpt.pt', recursive=True))
if not prev:
    raise SystemExit('Attach a previous output as an input: it must contain work/runs/small_wiki/ckpt.pt')
def last_step(ckpt):                      # from the run log, without loading the checkpoint
    log = os.path.join(os.path.dirname(ckpt), 'log.jsonl')
    try:
        return max(json.loads(l)['step'] for l in open(log) if l.strip())
    except Exception:
        return 0
prev.sort(key=last_step)                  # several outputs attached -> the most trained small_wiki wins
src = prev[-1].split('/work/')[0]
print('previous output:', src, '| small_wiki step', last_step(prev[-1]))
import torch
if not os.path.exists('work/bpe16000'):
    shutil.copytree(f'{src}/work/bpe16000', 'work/bpe16000')                      # tokenizer + token shards (~0.6 GB)
if not os.path.exists('work/runs/small_wiki/ckpt.pt'):
    shutil.copytree(f'{src}/work/runs/small_wiki', 'work/runs/small_wiki')         # with optimizer state: needed to resume
for r in ['tiny', 'tiny_upos']:                                                     # weights only: just for the per-POS chart
    p = f'{src}/work/runs/{r}/ckpt.pt'
    if os.path.exists(p) and not os.path.exists(f'work/runs/{r}/ckpt.pt'):
        st = torch.load(p, map_location='cpu', weights_only=False); st['optimizer'] = None
        os.makedirs(f'work/runs/{r}', exist_ok=True); torch.save(st, f'work/runs/{r}/ckpt.pt')
for d in ['work/bm25', 'work/teacher']:                                             # reusable stage-2 products, if the input has them
    if os.path.exists(f'{src}/{d}') and not os.path.exists(d):
        shutil.copytree(f'{src}/{d}', d)
for f in glob.glob(f'{src}/results/*.json'):
    shutil.copy(f, 'results/')
for f in ['simplewiki.jsonl', 'enwiki.jsonl']:                                      # raw text, if a previous stage-2 run kept it
    if os.path.exists(f'{src}/data/{f}') and not os.path.exists(f'data/{f}'):
        shutil.copy(f'{src}/data/{f}', 'data/')
from gramlab.amp import pick_precision
print('checkpoints:', sorted(glob.glob('work/runs/*/ckpt.pt')))
print('torch', torch.__version__, '| cuda', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
      '| training precision', pick_precision('cuda' if torch.cuda.is_available() else 'cpu')[4])

## 1. Wikipedia text (fetched once; from now on it stays in the output)

In [ ]:
if not os.path.exists('data/simplewiki.jsonl'):
    sh('python scripts/fetch_wikipedia.py --lang simple --out data/simplewiki.jsonl')
if not os.path.exists('data/enwiki.jsonl'):
    sh('python scripts/fetch_wikipedia.py --lang en --max_articles 120000 --out data/enwiki.jsonl')
sh('python -m gramlab.prepare --config configs/small_wiki.json')      # shards exist -> only verifies
WIKI = '--wiki data/simplewiki.jsonl --wiki data/enwiki.jsonl'
print(f'elapsed {(time.time()-T0)/60:.0f} min')

## 2. Finish the 25M writer in fp16 — `MORE_STEPS` more steps, re-warmed cosine fitted into `TRAIN_HOURS`

In [ ]:
if MORE_STEPS == 0:
    print('MORE_STEPS = 0: keeping the writer as it is')
extra = ' --set train.eval_every=100 --set train.eval_tokens=100000' if QUICK else ''
if MORE_STEPS: sh(f'python -m gramlab.train --config configs/small_wiki.json --resume --more_steps {300 if QUICK else MORE_STEPS} '
   f'--set train.lr=3e-4 --set train.max_hours={0.25 if QUICK else TRAIN_HOURS}' + extra)
print('\neval history:')
for line in open('work/runs/small_wiki/log.jsonl'):
    r = json.loads(line); print({k: r[k] for k in ('step', 'train_lm_loss', 'eval_loss', 'eval_ppl', 'eval_bits_per_byte', 'tokens_seen')})
print(f'elapsed {(time.time()-T0)/60:.0f} min')

## 3. BM25 index + the retrieval scoreboard (recall@k on title and body-sentence queries — the number the meaning map must beat)

In [ ]:
if not os.path.exists('work/bm25/weights.npz'):
    sh(f'python -m gramlab.retrieval build {WIKI} --out work/bm25 --max_words 100')
if not os.path.exists('results/retrieval_bm25.json'):
    sh(f'python -m gramlab.retrieval eval {WIKI} --index work/bm25 --n {400 if QUICK else 3000} --out results/retrieval_bm25.json')
for q in ['Tell me about Belgrade.', 'Who was Napoleon?', 'What is photosynthesis?']:
    sh(f'python -m gramlab.retrieval search --index work/bm25 --q "{q}" -k 2', must=False)

## 4. Retrieval-conditioned Q&A → `small_wiki_rag`; plain Q&A → `small_wiki_sft` (the no-retrieval comparison)

In [ ]:
sh(f'python scripts/make_rag_qa.py {WIKI} --index work/bm25 --out data/rag_qa.jsonl --n {5000 if QUICK else 30000}')
sh(f'python -m gramlab.finetune --init_from work/runs/small_wiki/ckpt.pt --corpus rag_qa --out small_wiki_rag --epochs {1 if QUICK else 2}')
if not os.path.exists('data/wiki_qa.jsonl'):
    sh('python scripts/make_wiki_qa.py --amalgum external/amalgum/amalgum --out data/wiki_qa.jsonl')
sh(f'python -m gramlab.finetune --init_from work/runs/small_wiki/ckpt.pt --corpus wiki_qa --out small_wiki_sft --epochs {1 if QUICK else 3}', must=False)
print(f'elapsed {(time.time()-T0)/60:.0f} min')

## 5. Teacher embeddings — targets for the grammar meaning map, plus a dense-retrieval baseline on the same queries
all-MiniLM-L6-v2 over every BM25 passage and every parsed AMALGUM sentence (~10 min on a T4), STS-B pairs + the teacher's Spearman.
The map stage (CPU) then regresses 32-dim slot codes from the chunked sentences onto `work/teacher/amalgum_sent.npy`.

In [ ]:
try:
    import sentence_transformers
except ImportError:
    sh('pip install -q sentence-transformers', must=False)
if not os.path.exists('work/teacher/amalgum_sent.npy'):
    sh('python scripts/teacher_embed.py --index work/bm25 --queries work/bm25/eval_queries.jsonl --scoreboard results/retrieval_dense.json '
          '--amalgum external/amalgum/amalgum --stsb data/stsb.jsonl --out work/teacher', must=False)
for f in ['results/retrieval_bm25.json', 'results/retrieval_dense.json', 'results/stsb_teacher.json']:
    if os.path.exists(f):
        r = json.load(open(f)); print(f, json.dumps({k: r[k] for k in r if k in ('all', 'title', 'body', 'teacher_spearman', 'system')}))
print(f'elapsed {(time.time()-T0)/60:.0f} min')

## 5b. The grammar-heads test at small data: 3 arms x 3 seeds x 250k / 1M / 4M tokens
Run 1 compared the arms after 33M tokens (9 passes over AMALGUM); the CPU n-gram curve says the POS-class gain shrinks with data
(-13.6% perplexity at 100k tokens, -8% at 3.5M), so that comparison could only come out null. This is the sample-efficiency test
the hypothesis actually makes: if `upos` beats `none` by more than the seed spread at 250k-1M and the arms converge at 4M, grammar
buys data at zero inference cost. Checkpoints are deleted after evaluation.

In [ ]:
if SWEEP:
    sh(f"python scripts/sweep_arms.py --config configs/tiny.json --tokens {'250000' if QUICK else '250000 1000000 4000000'} --seeds {'1' if QUICK else '1 2 3'} "
       f"--arms none upos chunk --blimp external/blimp/data --blimp_limit {30 if QUICK else 100} --out results/sweep.json --plot results/sweep.png", must=False)
    if os.path.exists('results/sweep.png'):
        from IPython.display import Image, display
        display(Image('results/sweep.png'))
print(f'elapsed {(time.time()-T0)/60:.0f} min')

## 6. Per-POS loss on gold GUM for `tiny`, `tiny_upos`, `small_wiki`: grammar words learned, names not

In [ ]:
runs = [r for r in ['tiny', 'tiny_upos', 'small_wiki'] if os.path.exists(f'work/runs/{r}/ckpt.pt')]
for r in runs:
    sh(f'python -m gramlab.evaluate --ckpt work/runs/{r}/ckpt.pt --corpus gum_test --per_pos --out results/eval_{r}.json > results/eval_{r}.log', must=False)
done = [r for r in runs if os.path.exists(f'results/eval_{r}.json')]
for r in done:
    e = json.load(open(f'results/eval_{r}.json'))
    print(f"{r:11} ppl {e['eval_ppl']:.1f}  bits/byte {e['eval_bits_per_byte']:.4f}  tagged-token loss {e['tagged_loss']:.3f}  chunk start/inside/outside "
          + '/'.join(f"{e['per_chunk_position'][k]['loss']:.2f}" for k in ('B', 'I', 'O') if k in e['per_chunk_position']))
sh('python scripts/plot_pos_loss.py --results ' + ' '.join(f'results/eval_{r}.json' for r in done) + ' --out results/per_pos_loss.png')
from IPython.display import Image, display
display(Image('results/per_pos_loss.png'))

## 7. Same questions with retrieval (`small_wiki_rag` + BM25) and without (`small_wiki_sft`)

In [ ]:
import torch
sys.path.insert(0, '/kaggle/working/gramlab')
from gramlab.ckpt import load_checkpoint
from gramlab.tokenizer import USER, ASSISTANT, END, EOS
from gramlab.retrieval import BM25Index, format_rag_user
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
index = BM25Index.load('work/bm25')
models = {n: load_checkpoint(f'work/runs/{n}/ckpt.pt', dev) for n in ['small_wiki_rag', 'small_wiki_sft'] if os.path.exists(f'work/runs/{n}/ckpt.pt')}

def answer(name, user_text, temperature=0.5, top_k=40, top_p=0.9, rep=1.2, max_new=80):
    model, tok, state = models[name]
    ids = [USER] + tok.encode(user_text) + [END, ASSISTANT]
    x = torch.tensor([ids[-(model.ctx - 1):]], dtype=torch.long, device=dev)
    out = model.generate(x, max_new_tokens=max_new, temperature=temperature, top_k=top_k, top_p=top_p, repetition_penalty=rep, stop_ids=(END, EOS))
    return tok.decode(out[0, x.size(1):].tolist())

for q in ['Tell me about Belgrade.', 'Tell me about Paris.', 'Who was Napoleon?', 'What is the capital of Japan?', 'Tell me about the Danube.',
          'How do I bake bread?', 'Who was Albert Einstein?', 'What is photosynthesis?', 'What is a volcano?', 'Tell me about Zxqv Blorptown.']:
    passages = index.retrieve(q, k=1, max_words=90)
    print('Q:', q)
    for p in passages: print(f"   retrieved [{p['score']:.1f}] {p['title']}: {p['text'][:140]}...")
    if 'small_wiki_rag' in models: print('   with retrieval >', answer('small_wiki_rag', format_rag_user(passages or [{'title': '(no match)', 'text': 'No passage found.'}], q)))
    if 'small_wiki_sft' in models: print('   no retrieval   >', answer('small_wiki_sft', q))
    print()

## 8. Trim: git clones go, everything reusable stays (raw text, shards, checkpoints, BM25, teacher embeddings, Q&A sets, results)

In [ ]:
for d in ['external', '.git']:
    shutil.rmtree(os.path.join('/kaggle/working/gramlab', d), ignore_errors=True)
os.system('du -sh /kaggle/working/gramlab/work/* /kaggle/working/gramlab/data /kaggle/working/gramlab/results 2>/dev/null')
print(f'total run time {(time.time()-T0)/3600:.2f} h')

## 9. Chat — talk to the current model yourself
Pick `MODE` and type in the box (or call `chat('...')`). Self-contained: it runs here right after the pipeline, **or** on its own in a
fresh interactive session with this notebook's latest output attached as an input (*Add Input → Your Work → Notebooks*) — nothing
above needs to run first. `rag` = `small_wiki_rag` answers from the BM25 passage it is shown (the line in brackets); `sft` = plain Q&A,
no retrieval, 3 turns of history; `base` = pretrained `small_wiki` continues your text.

In [ ]:
import os, sys, glob, subprocess, torch
os.chdir('/kaggle/working')
if not os.path.exists('gramlab/gramlab'):                                   # fresh session: only the code is needed
    subprocess.run('git clone --depth 1 https://github.com/furevych/gramlab.git', shell=True, check=True)
os.chdir('/kaggle/working/gramlab'); sys.path.insert(0, '/kaggle/working/gramlab')
from gramlab.ckpt import load_checkpoint
from gramlab.tokenizer import USER, ASSISTANT, END, EOS
from gramlab.retrieval import BM25Index, format_rag_user

def find(rel):                      # this run's work/ first, else the attached outputs (newest attached version last in the list)
    hits = [rel] if os.path.exists(rel) else sorted(glob.glob(f'/kaggle/input/**/{rel}', recursive=True), key=os.path.getmtime)
    return hits[-1] if hits else None
tokd = find('work/bpe16000/tokenizer')          # the checkpoint points at this relative path
if tokd and not os.path.exists('work/bpe16000/tokenizer'):
    os.makedirs('work/bpe16000', exist_ok=True); os.symlink(os.path.abspath(tokd), 'work/bpe16000/tokenizer')

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
MODELS = {}
for n in ['small_wiki_rag', 'small_wiki_sft', 'small_wiki']:
    p = find(f'work/runs/{n}/ckpt.pt')
    if p:
        MODELS[n] = load_checkpoint(p, dev); print(f'{n:15} step {MODELS[n][2]["step"]:>6}  <- {p}')
bm = find('work/bm25'); INDEX = BM25Index.load(bm) if bm else None
print('BM25 index:', bm or 'not found (rag mode will answer without a passage)')

MODE = 'rag'          # 'rag' | 'sft' | 'base'
history = []
def chat(text, temperature=0.5, top_k=40, top_p=0.9, rep=1.2, max_new=100, k=1):
    global history
    name = {'rag': 'small_wiki_rag', 'sft': 'small_wiki_sft', 'base': 'small_wiki'}[MODE]
    if name not in MODELS:
        return f'(no checkpoint for {name}; have: {list(MODELS)})'
    model, tok, _ = MODELS[name]
    if MODE == 'base':
        ids, stop = tok.encode(text), ()
    else:
        if MODE == 'rag':
            passages = INDEX.retrieve(text, k=k, max_words=90) if INDEX else []
            print('   [' + ('; '.join(f"{p['title']} {p['score']:.1f}" for p in passages) or 'no passage') + ']')
            turns = [('user', format_rag_user(passages or [{'title': '(no match)', 'text': 'No passage found.'}], text))]
        else:
            history.append(('user', text)); turns = history[-6:]
        ids = []
        for role, t in turns:
            ids += [USER if role == 'user' else ASSISTANT] + tok.encode(t) + [END]
        ids += [ASSISTANT]; stop = (END, EOS)
    x = torch.tensor([ids[-(model.ctx - 1):]], dtype=torch.long, device=dev)
    out = model.generate(x, max_new_tokens=max_new, temperature=temperature, top_k=top_k, top_p=top_p, repetition_penalty=rep, stop_ids=stop)
    reply = tok.decode(out[0, x.size(1):].tolist())
    if MODE == 'sft':
        history.append(('assistant', reply))
    return reply

for q in ['Tell me about Belgrade.', 'What is photosynthesis?']:      # so the saved version shows the model talking
    print('you>  ', q); print('model>', chat(q)); print()

import ipywidgets as w
from IPython.display import display
box = w.Text(placeholder=f'ask the model ({MODE} mode)...', layout=w.Layout(width='70%'), continuous_update=False)
btn = w.Button(description='Send'); out = w.Output()
def on_send(_=None):
    text = box.value.strip()
    if not text: return
    box.value = ''
    with out:
        print('you>  ', text); print('model>', chat(text)); print()
btn.on_click(on_send)
box.observe(lambda ch: on_send() if ch['name'] == 'value' and ch['new'] else None, names='value')
display(w.HBox([box, btn]), out)
